# Huan luyen va danh gia mo hinh LightGBM

Huan luyen mo hinh LightGBM tren clean_base.csv voi phan chia Out-Of-Time (Train thang 0-5, Val thang 6, Test thang 7).


In [ ]:
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.models.train_model import prepare_datasets, train_lightgbm_model, plot_curves, plot_feature_importance
from src.models.evaluate import evaluate_fraud_model, print_evaluation_summary


### Nap va phan chia du lieu Out-Of-Time


In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = prepare_datasets("../data/processed/clean_base.csv")

### Nap hoac huan luyen mo hinh LightGBM


In [ ]:
model_path = Path("../models/lgbm_fraud_model.joblib")
if model_path.exists():
    print("Đang nạp mô hình đã train sẵn...")
    model = joblib.load(model_path)
else:
    print("Huấn luyện mô hình mới...")
    model = train_lightgbm_model(X_train, y_train, X_val, y_val)


### Danh gia hieu nang tren tap Test OOT (Thang 7)


In [ ]:
test_probs = model.predict_proba(X_test)[:, 1]
metrics = evaluate_fraud_model(y_test, test_probs, dataset_name="Test Set OOT (Tháng 7)")
print_evaluation_summary(metrics)

### Top dac trung quan trong nhat


In [ ]:
feat_imp = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, x="importance", y="feature", hue="feature", palette="Blues_r", legend=False)
plt.title("Top 15 Đặc Trưng Quan Trọng Nhất (LightGBM)")
plt.xlabel("Số lần phân nhánh (Split Count)")
plt.show()